# 07 — Predictions & Mapping

Generación de mapas de turbidez superficial sobre el polígono de Mar Menor.

**Flujo:**
1. Cargar el mejor modelo entrenado en notebook 05
2. Seleccionar una imagen satelital de la fecha de test más reciente disponible
3. Crear una malla regular (200 m) dentro del polígono
4. Extraer las mismas features que en entrenamiento (buffer circular de radio según dataset)
5. Predecir turbidez (escala original via expm1)
6. Visualizar sobre imagen RGB de satélite con el polígono enmascarado
7. Comparar XGBoost vs CatBoost sobre el mismo mapa

**Requiere:** haber ejecutado `05_model_training.ipynb` primero.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize
from pathlib import Path
import pickle
import json
import re
import warnings
warnings.filterwarnings('ignore')

import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.transform import rowcol, array_bounds

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
BASE_DIR = Path('..')
SAT_DIR  = BASE_DIR / 'data' / 'Satellite'
GEO_DIR  = BASE_DIR / 'geospatial'
RES_DIR  = BASE_DIR / 'results'
MOD_DIR  = RES_DIR  / 'models'

BAND_NAMES = ['coastal', 'blue', 'green_i', 'green', 'yellow', 'red', 'red_edge', 'nir']
RES_M      = 3      # resolución del TIF en m/px
GRID_STEP  = 200    # separación de la malla de predicción en metros
MIN_VALID  = 50     # píxeles válidos mínimos por buffer

MODEL_COLORS = {'XGBoost': '#ED7D31', 'CatBoost': '#4472C4'}

## 1. Cargar modelos y configuración

In [ ]:
with open(RES_DIR / 'best_model_info.json') as f:
    best_info = json.load(f)
with open(RES_DIR / 'dataset_config.json') as f:
    dataset_config = json.load(f)

best_ds  = best_info['best_dataset']
best_mod = best_info['best_model']
radius_px = int(best_info['radius_px'])

print(f"Mejor dataset  : {best_ds}  (diámetro extracción = {best_info['diameter_m']} m)")
print(f"Mejor modelo   : {best_mod}")
print(f"Radio buffer   : {radius_px} px = {radius_px * RES_M} m")

# Cargar el mejor modelo y sus feature columns
best_model_path = MOD_DIR / f'{best_ds}__{best_mod}.pkl'
feat_cols_path  = MOD_DIR / f'{best_ds}__feat_cols.pkl'

best_model = pickle.load(open(best_model_path, 'rb'))
feat_cols  = pickle.load(open(feat_cols_path,  'rb'))

# Cargar también el segundo modelo (para comparar)
available_models = [f.stem.replace(f'{best_ds}__', '')
                    for f in MOD_DIR.glob(f'{best_ds}__*.pkl')
                    if '__feat_cols' not in f.stem]

models_map = {}
for mname in available_models:
    mpath = MOD_DIR / f'{best_ds}__{mname}.pkl'
    if mpath.exists():
        models_map[mname] = pickle.load(open(mpath, 'rb'))
        print(f"Modelo cargado : {mname}")

## 2. Seleccionar imagen satelital (fecha de test)

In [ ]:
# Leer las fechas de test desde el CSV de predicciones
test_preds_df = pd.read_csv(RES_DIR / 'test_predictions.csv', parse_dates=['date'])
test_dates    = sorted(test_preds_df['date'].unique())

# Inventario de TIFs con su fecha
date_pat  = re.compile(r'(\d{4}-\d{2}-\d{2})')
tif_files = sorted(SAT_DIR.glob('*_composite.tif'))  # sin _udm2
tif_by_date = {}
for f in tif_files:
    m = date_pat.search(f.stem)
    if m:
        tif_by_date[pd.Timestamp(m.group(1))] = f

# Buscar la fecha de test más reciente con TIF disponible
map_date   = None
target_tif = None
for d in reversed(test_dates):
    if d in tif_by_date:
        map_date, target_tif = d, tif_by_date[d]
        break

# Fallback: TIF más cercano
if target_tif is None:
    for d in reversed(test_dates):
        avail = sorted(tif_by_date.keys(), key=lambda x: abs((x - d).days))
        if avail and abs((avail[0] - d).days) <= 7:
            map_date, target_tif = d, tif_by_date[avail[0]]
            break

if target_tif is None:
    map_date   = test_dates[-1]
    target_tif = list(tif_by_date.values())[-1]
    print("AVISO: usando imagen de fallback")

print(f"Fecha del mapa : {pd.Timestamp(map_date).date()}")
print(f"Imagen TIF     : {target_tif.name}")

## 3. Cargar imagen y crear compuesto RGB

In [ ]:
# ── Cargar geometrías ─────────────────────────────────────────────────────────
gdf_poly = gpd.read_file(GEO_DIR / 'marmenor_polygon.geojson')  # EPSG:32630
ctd_gdf  = gpd.read_file(GEO_DIR / 'ctd_points.geojson').to_crs(gdf_poly.crs)
polygon  = gdf_poly.geometry.unary_union

# ── Cargar imagen recortada al polígono (+ buffer para los puntos de borde) ───
extra_buf = radius_px * RES_M + 10   # margen extra en metros

yy, xx    = np.ogrid[-radius_px:radius_px+1, -radius_px:radius_px+1]
circ_mask = (xx**2 + yy**2) <= radius_px**2   # máscara circular

with rasterio.open(target_tif) as src:
    # Asegurar mismo CRS
    if gdf_poly.crs.to_epsg() != src.crs.to_epsg():
        clip_poly = gdf_poly.to_crs(src.crs).geometry.unary_union
    else:
        clip_poly = polygon
    clip_geom  = [clip_poly.buffer(extra_buf).__geo_interface__]
    img_clip, clip_tf = rio_mask(src, clip_geom, crop=True, nodata=0, filled=True)
    raster_crs = src.crs

img_clip = img_clip.astype(np.float32)
img_clip[img_clip == 0] = np.nan   # nodata → NaN

n_bands, img_h, img_w = img_clip.shape
print(f"Imagen recortada : {n_bands} bandas × {img_h} × {img_w} px")
print(f"Radio buffer     : {radius_px} px = {radius_px * RES_M} m")

In [ ]:
def pct_stretch(band, p_lo=2, p_hi=98):
    """Normalización por percentiles para visualización RGB."""
    valid = band[band > 0]
    if valid.size == 0:
        return np.zeros_like(band, dtype=np.float32)
    lo, hi = np.percentile(valid, [p_lo, p_hi])
    return np.clip((band.astype(np.float32) - lo) / (hi - lo + 1e-6), 0, 1)

# RGB: red (idx 5) — green (idx 3) — blue (idx 1)
rgb = np.stack([
    pct_stretch(img_clip[5]),
    pct_stretch(img_clip[3]),
    pct_stretch(img_clip[1]),
], axis=-1)

# Extent del recorte para imshow
# array_bounds devuelve (west, south, east, north) = (left, bottom, right, top)
left, bottom, right, top = array_bounds(img_h, img_w, clip_tf)
extent_img = [left, right, bottom, top]

print(f"Compuesto RGB : {rgb.shape}")
print(f"Extent (m)    : x=[{left:.0f}, {right:.0f}]  y=[{bottom:.0f}, {top:.0f}]")

## 4. Crear malla de predicción dentro del polígono

In [ ]:
bds = polygon.bounds  # (minx, miny, maxx, maxy)
xs  = np.arange(bds[0] + GRID_STEP/2, bds[2], GRID_STEP)
ys  = np.arange(bds[1] + GRID_STEP/2, bds[3], GRID_STEP)
gx, gy = np.meshgrid(xs, ys)
all_coords = np.column_stack([gx.ravel(), gy.ravel()])

grid_gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(all_coords[:, 0], all_coords[:, 1]),
    crs=gdf_poly.crs
)
in_poly     = grid_gdf.geometry.within(polygon).values
grid_coords = all_coords[in_poly]
print(f"Puntos de malla dentro del polígono: {len(grid_coords)}  (paso {GRID_STEP} m)")

## 5. Extraer features en cada punto de la malla

In [ ]:
def band_stats(pixels_1d):
    """mean, median, std, p10, p90 — mismas estadísticas que en entrenamiento."""
    return [
        np.nanmean(pixels_1d),
        np.nanmedian(pixels_1d),
        np.nanstd(pixels_1d),
        np.nanpercentile(pixels_1d, 10),
        np.nanpercentile(pixels_1d, 90),
    ]

def spectral_indices(means):
    """Índices espectrales a partir de medias de banda (mismos que feat engineering)."""
    c, b, gi, g, ye, r, re, n = means
    eps = 1.0   # escala uint16 → ratios son invariantes al escalado
    return [
        (n - g)  / (n + g  + eps),   # ndti
        r        / (g      + eps),   # red_green
        b        / (g      + eps),   # blue_green
        c        / (g      + eps),   # coastal_green
        (re - r) / (re + r + eps),   # rededge_idx
    ]


n_pts      = len(grid_coords)
n_feat     = len(feat_cols)
feats_grid = np.full((n_pts, n_feat), np.nan, dtype=np.float64)
valid_pts  = np.zeros(n_pts, dtype=bool)

print(f"Extrayendo features en {n_pts} puntos (radio={radius_px}px)...")

for i, (gx_pt, gy_pt) in enumerate(grid_coords):
    if i % 400 == 0:
        print(f"  {i}/{n_pts}", end='\r')

    row_c, col_c = rowcol(clip_tf, gx_pt, gy_pt)

    r0 = max(0, row_c - radius_px);  r1 = min(img_h, row_c + radius_px + 1)
    c0 = max(0, col_c - radius_px);  c1 = min(img_w, col_c + radius_px + 1)
    if r0 >= r1 or c0 >= c1:
        continue

    # Recorte de la máscara circular para ventanas en el borde de la imagen
    cm_r0 = radius_px - (row_c - r0);  cm_r1 = cm_r0 + (r1 - r0)
    cm_c0 = radius_px - (col_c - c0);  cm_c1 = cm_c0 + (c1 - c0)
    cm_sub = circ_mask[cm_r0:cm_r1, cm_c0:cm_c1]

    window = img_clip[:, r0:r1, c0:c1].copy()
    window[:, ~cm_sub] = np.nan

    if np.sum(~np.isnan(window[0])) < MIN_VALID:
        continue

    row_feats  = []
    band_means = []
    for b in range(n_bands):
        st = band_stats(window[b].ravel())
        row_feats.extend(st)
        band_means.append(st[0])

    row_feats.extend(spectral_indices(band_means))
    feats_grid[i] = row_feats
    valid_pts[i]  = True

coords_valid = grid_coords[valid_pts]
feats_valid  = feats_grid[valid_pts]
print(f"\nPuntos válidos: {valid_pts.sum()} / {n_pts}  "
      f"({valid_pts.sum()/n_pts*100:.1f}%)")

## 6. Predicción de turbidez

In [ ]:
# Predecir con todos los modelos disponibles para el mejor dataset
preds_map = {}
for mname, model in models_map.items():
    raw_pred = model.predict(feats_valid)
    preds_map[mname] = np.maximum(np.expm1(raw_pred), 0.0)
    print(f"{mname:<12} media={preds_map[mname].mean():.3f} NTU  "
          f"rango=[{preds_map[mname].min():.2f}, {preds_map[mname].max():.2f}]")

# Valores observados en la fecha del mapa (estaciones CTD)
date_obs = test_preds_df[test_preds_df['date'] == map_date][['ctd','turbidity']]
print(f"\nObservaciones CTD el {pd.Timestamp(map_date).date()}:")
print(date_obs.to_string(index=False))

## 7. Mapa de turbidez sobre imagen satelital

In [ ]:
ctd_name_col = 'ControlPointCode'

# CRS unificado para todos los elementos geoespaciales
if gdf_poly.crs.to_epsg() != raster_crs.to_epsg():
    gdf_poly_r = gdf_poly.to_crs(raster_crs)
    ctd_gdf_r  = ctd_gdf.to_crs(raster_crs)
else:
    gdf_poly_r = gdf_poly
    ctd_gdf_r  = ctd_gdf

# Escala de color compartida entre modelos
all_preds_concat = np.concatenate(list(preds_map.values()))
vmin_t = 0
vmax_t = np.percentile(all_preds_concat, 97)
cmap_t = plt.cm.YlOrRd
norm_t = Normalize(vmin=vmin_t, vmax=vmax_t)

n_models   = len(preds_map)
n_cols_fig = n_models + 1   # columna extra para RGB

fig, axes = plt.subplots(1, n_cols_fig,
                          figsize=(7 * n_cols_fig, 8))

# ── Panel 1: imagen RGB ────────────────────────────────────────────────────────
ax = axes[0]
ax.imshow(rgb, extent=extent_img, origin='upper', aspect='equal', zorder=1)
gdf_poly_r.boundary.plot(ax=ax, color='white', linewidth=2.0, zorder=5)
ctd_gdf_r.plot(ax=ax, color='cyan', markersize=50, zorder=6,
               edgecolor='navy', linewidth=1.2, marker='^')
ax.set_xlim(left, right);  ax.set_ylim(bottom, top)
ax.set_title(f'Imagen satelital RGB\nPlanet SuperDove — {pd.Timestamp(map_date).date()}',
             fontsize=11)
ax.set_xlabel('Easting (m)');  ax.set_ylabel('Northing (m)')

# ── Paneles de modelos: turbidez sobre imagen ──────────────────────────────────
for ax, (mname, preds) in zip(axes[1:], preds_map.items()):
    ax.imshow(rgb, extent=extent_img, origin='upper', aspect='equal', zorder=1)

    sc = ax.scatter(
        coords_valid[:, 0], coords_valid[:, 1],
        c=preds, cmap=cmap_t, norm=norm_t,
        s=18, alpha=0.82, linewidths=0, zorder=3
    )

    gdf_poly_r.boundary.plot(ax=ax, color='white', linewidth=2.0, zorder=5)
    ctd_gdf_r.plot(ax=ax, color='cyan', markersize=50, zorder=6,
                   edgecolor='navy', linewidth=1.2, marker='^')

    # Anotar turbidez observada en cada CTD para esa fecha
    for _, obs in date_obs.iterrows():
        rows = ctd_gdf_r[ctd_gdf_r[ctd_name_col] == obs['ctd']]
        if len(rows):
            pt = rows.geometry.values[0]
            ax.annotate(
                f"{obs['turbidity']:.1f}",
                xy=(pt.x, pt.y), xytext=(8, 6),
                textcoords='offset points',
                fontsize=8.5, fontweight='bold', color='white',
                bbox=dict(boxstyle='round,pad=0.25', fc='#003399',
                          alpha=0.75, ec='none')
            )

    cbar = plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.03)
    cbar.set_label('Turbidez estimada (NTU)')
    ax.set_xlim(left, right);  ax.set_ylim(bottom, top)
    ax.set_title(
        f'{mname} — Turbidez superficial\n'
        f'{pd.Timestamp(map_date).date()}  |  dataset: {best_ds}',
        fontsize=11
    )
    ax.set_xlabel('Easting (m)')

plt.suptitle(
    'Mar Menor — Estimación de turbidez superficial sobre imagen satelital',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(RES_DIR / 'turbidity_on_satellite.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Comparación espacial: XGBoost vs CatBoost

In [ ]:
model_names = list(preds_map.keys())

if len(model_names) < 2:
    print("Solo hay un modelo disponible; se omite el panel de diferencia.")
else:
    mn1, mn2 = model_names[0], model_names[1]
    diff = preds_map[mn1] - preds_map[mn2]
    v_abs = np.percentile(np.abs(diff), 97)

    fig, axes = plt.subplots(1, 3, figsize=(21, 7))

    for ax, (mname, preds) in zip(axes[:2], preds_map.items()):
        ax.imshow(rgb, extent=extent_img, origin='upper', aspect='equal', zorder=1)
        sc = ax.scatter(coords_valid[:, 0], coords_valid[:, 1],
                        c=preds, cmap=cmap_t, norm=norm_t,
                        s=18, alpha=0.84, linewidths=0, zorder=3)
        gdf_poly_r.boundary.plot(ax=ax, color='white', linewidth=1.8, zorder=5)
        plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.03, label='Turbidez (NTU)')
        ax.set_xlim(left, right);  ax.set_ylim(bottom, top)
        ax.set_title(mname, fontsize=12)
        ax.set_xlabel('Easting (m)')
        ax.set_ylabel('Northing (m)')

    # Diferencia
    ax = axes[2]
    ax.imshow(rgb, extent=extent_img, origin='upper', aspect='equal', zorder=1)
    sc3 = ax.scatter(coords_valid[:, 0], coords_valid[:, 1],
                     c=diff, cmap='RdBu_r',
                     vmin=-v_abs, vmax=v_abs,
                     s=18, alpha=0.84, linewidths=0, zorder=3)
    gdf_poly_r.boundary.plot(ax=ax, color='white', linewidth=1.8, zorder=5)
    cbar3 = plt.colorbar(sc3, ax=ax, fraction=0.035, pad=0.03)
    cbar3.set_label(f'{mn1} − {mn2} (NTU)')
    ax.set_xlim(left, right);  ax.set_ylim(bottom, top)
    ax.set_title(f'Diferencia {mn1} − {mn2}', fontsize=12)
    ax.set_xlabel('Easting (m)')

    plt.suptitle(f'Comparación de modelos — {pd.Timestamp(map_date).date()}',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(RES_DIR / 'comparison_models_map.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Diferencia media {mn1}−{mn2}  : {diff.mean():+.3f} NTU")
    print(f"MAE espacial entre modelos    : {np.abs(diff).mean():.3f} NTU")

## 9. Guardar predicciones de malla

In [ ]:
grid_out = pd.DataFrame({'x': coords_valid[:, 0], 'y': coords_valid[:, 1]})
for mname, preds in preds_map.items():
    grid_out[f'pred_{mname}'] = preds
grid_out['map_date'] = str(pd.Timestamp(map_date).date())

out_path = RES_DIR / 'grid_turbidity_predictions.csv'
grid_out.to_csv(out_path, index=False)

print(f"Predicciones de malla guardadas en: {out_path}")
print(f"  Filas: {len(grid_out)}  |  Columnas: {list(grid_out.columns)}")
print()
print("Archivos de mapas guardados en results/:")
print("  turbidity_on_satellite.png")
print("  comparison_models_map.png")
print("  grid_turbidity_predictions.csv")

## 10. Mapa solo turbidez (alta resolución para exportar)

In [ ]:
# Mapa limpio del mejor modelo: solo turbidez sobre fondo blanco + contorno del polígono
best_preds = preds_map[best_mod] if best_mod in preds_map else list(preds_map.values())[0]

fig, ax = plt.subplots(figsize=(9, 11))

gdf_poly_r.plot(ax=ax, color='#f0f5ff', edgecolor='navy', linewidth=1.5, zorder=1)
sc = ax.scatter(
    coords_valid[:, 0], coords_valid[:, 1],
    c=best_preds, cmap='YlOrRd', vmin=0, vmax=vmax_t,
    s=22, alpha=0.95, linewidths=0, zorder=2
)
ctd_gdf_r.plot(ax=ax, color='white', markersize=70, zorder=4,
               edgecolor='navy', linewidth=1.5, marker='^')
# Etiquetas de valores observados
for _, obs in date_obs.iterrows():
    rows = ctd_gdf_r[ctd_gdf_r[ctd_name_col] == obs['ctd']]
    if len(rows):
        pt = rows.geometry.values[0]
        ax.annotate(
            f"{obs['ctd']}\n{obs['turbidity']:.1f} NTU",
            xy=(pt.x, pt.y), xytext=(10, 8),
            textcoords='offset points',
            fontsize=7.5, color='navy',
            bbox=dict(boxstyle='round,pad=0.25', fc='white', alpha=0.7, ec='none')
        )

cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Turbidez (NTU)', fontsize=12)
ax.set_title(
    f'Mar Menor — Turbidez superficial estimada\n'
    f'{best_mod} | dataset: {best_ds} | {pd.Timestamp(map_date).date()}',
    fontsize=12
)
ax.set_xlabel('Easting (m)');  ax.set_ylabel('Northing (m)')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(RES_DIR / 'turbidity_map_clean.png', dpi=200, bbox_inches='tight')
plt.show()